### Process All County Station Data and Identify Heat Events

This script automatically processes every county CSV in the `Station_Data` folder, filters stations with excessive missing data, calculates daily average maximum temperatures, identifies consecutive extreme heat events using both an absolute (>90°F) and county-specific (95th percentile) threshold, saves the resulting date ranges to county-specific output folders, and displays the top 10 longest heat events for each threshold.

In [21]:
import os
import pandas as pd
from IPython.display import display

In [22]:
DATA_FOLDER = "Station_Data"
OUTPUT_FOLDER = "Heat_Events"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def get_consecutive_ranges(df):
    """Identify consecutive date ranges."""
    if df.empty:
        return pd.DataFrame(columns=[
            "begin_date",
            "end_date",
            "duration",
            "avg_temp"
        ])

    df = df.sort_values("DATE").reset_index(drop=True)
    df["group"] = df["DATE"].diff().dt.days.ne(1).cumsum()

    ranges = (
        df.groupby("group")
        .agg(
            begin_date=("DATE", "first"),
            end_date=("DATE", "last"),
            duration=("DATE", "count"),
            avg_temp=("DAILY_TMAX_F", "mean"),
        )
        .reset_index(drop=True)
    )

    return ranges


In [23]:
for filename in sorted(os.listdir(DATA_FOLDER)):

    if not filename.endswith(".csv"):
        continue

    county = os.path.splitext(filename)[0]

    print("\n" + "=" * 70)
    print(f"{county.upper()}")
    print("=" * 70)

    file_path = os.path.join(DATA_FOLDER, filename)
    df = pd.read_csv(file_path)

    df["DATE"] = pd.to_datetime(df["DATE"])

    null_perc = df.groupby("STATION")["TMAX"].apply(lambda x: x.isnull().mean())

    good_stations = null_perc[null_perc <= 0.40].index

    df_clean = df[df["STATION"].isin(good_stations)].copy()

    print(
        f"Stations kept: {len(good_stations)} / {len(df['STATION'].unique())}"
    )

    daily_avg = (
        df_clean.groupby("DATE")["TMAX"]
        .mean()
        .reset_index(name="DAILY_TMAX_F")
    )

    percentile95 = daily_avg["DAILY_TMAX_F"].quantile(0.95)

    hot90 = daily_avg[daily_avg["DAILY_TMAX_F"] > 90].copy()
    hot95 = daily_avg[daily_avg["DAILY_TMAX_F"] > percentile95].copy()

    ranges90 = get_consecutive_ranges(hot90)
    ranges95 = get_consecutive_ranges(hot95)

    ranges90 = ranges90.sort_values(
        "duration",
        ascending=False
    )

    ranges95 = ranges95.sort_values(
        "duration",
        ascending=False
    )

    county_folder = os.path.join(OUTPUT_FOLDER, county)
    os.makedirs(county_folder, exist_ok=True)

    ranges90.to_csv(
        os.path.join(county_folder, "date_ranges_90f.csv"),
        index=False,
    )

    ranges95.to_csv(
        os.path.join(county_folder, "date_ranges_p95.csv"),
        index=False,
    )

    print("\nTop 10 >90°F Events")
    display(ranges90.head(10))

    print(f"\nTop 10 >95th Percentile ({percentile95:.1f}°F) Events")
    display(ranges95.head(10))

    print(f"\nSaved files to: {county_folder}")

print("\nAll counties processed.")


LA
Stations kept: 44 / 90

Top 10 >90°F Events


,begin_date,end_date,duration,avg_temp
1,2023-07-13,2023-07-31,19,93.273686
14,2024-08-31,2024-09-10,11,96.814050
7,2024-07-03,2024-07-11,9,94.098485
9,2024-07-18,2024-07-26,9,92.808962
15,2024-09-30,2024-10-07,8,92.244318
20,2025-08-06,2025-08-11,6,91.731061
23,2025-08-29,2025-09-03,6,92.159796
21,2025-08-20,2025-08-25,6,94.132576
10,2024-08-02,2024-08-07,6,93.878788
4,2023-08-27,2023-08-30,4,94.709302



Top 10 >95th Percentile (92.5°F) Events


,begin_date,end_date,duration,avg_temp
14,2024-09-03,2024-09-10,8,98.900568
10,2024-07-20,2024-07-25,6,93.628171
0,2023-07-14,2023-07-17,4,93.982558
17,2025-08-21,2025-08-24,4,95.426136
3,2023-07-24,2023-07-27,4,94.742940
6,2023-08-27,2023-08-30,4,94.709302
11,2024-08-03,2024-08-06,4,95.221591
15,2024-10-01,2024-10-03,3,93.795455
9,2024-07-09,2024-07-11,3,94.765152
8,2024-07-05,2024-07-07,3,95.340909



Saved files to: Heat_Events/LA

RIVERSIDE
Stations kept: 25 / 78

Top 10 >90°F Events


...


Saved files to: Heat_Events/San Diego

SAN FRANCISCO
Stations kept: 2 / 7

Top 10 >90°F Events


,begin_date,end_date,duration,avg_temp
0,2019-06-09,2019-06-11,3,94.166667
9,2024-10-05,2024-10-07,3,94.666667
3,2020-09-06,2020-09-07,2,95.500000
4,2020-10-16,2020-10-17,2,92.750000
6,2022-09-05,2022-09-06,2,94.500000
8,2024-10-01,2024-10-02,2,94.500000
1,2019-09-25,2019-09-25,1,94.500000
2,2020-08-14,2020-08-14,1,95.000000
5,2022-06-21,2022-06-21,1,92.000000
7,2023-10-05,2023-10-05,1,94.000000



Top 10 >95th Percentile (76.0°F) Events


,begin_date,end_date,duration,avg_temp
45,2024-09-30,2024-10-07,8,90.500000
8,2019-10-22,2019-10-29,8,81.187500
14,2020-08-13,2020-08-19,7,82.857143
33,2022-09-04,2022-09-08,5,87.800000
0,2019-06-09,2019-06-13,5,90.100000
5,2019-09-23,2019-09-26,4,83.875000
9,2019-10-31,2019-11-03,4,81.000000
20,2020-10-15,2020-10-18,4,89.250000
25,2021-09-30,2021-10-03,4,81.875000
6,2019-10-06,2019-10-08,3,82.500000



Saved files to: Heat_Events/San Francisco

All counties processed.
